# Zigbee Structured Signal

Generate and plot a Zigbee signal in a TorchSig dataset.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from torchsig.datasets import TorchSigIterableDataset
from torchsig.transforms.transforms import Spectrogram
from torchsig.transforms.metadata_transforms import YOLOLabel

### Configuration

In [ ]:
SEED = 1234567890                       # seed for random generators

# Scene variables
SAMPLE_RATE             = 10e6          # Hz
FFT_SIZE                = 512           # FFT dimension
NUM_IQ_SAMPLES          = 262144     # 262144 -> a 512x512 spectrogram at fft_size 512
SNR_DB                  = 50.0          # very high SNR so a signal dominates a scene
SIGNAL_DURATION_MIN     = NUM_IQ_SAMPLES * 0.8  # samples
SIGNAL_DURATION_MAX     = NUM_IQ_SAMPLES * 1.0  # samples
SIGNAL_BANDWIDTH_MIN    = 2500000       # Hz
SIGNAL_BANDWIDTH_MAX    = 3333333       # Hz

# dataset metadata
METADATA = {
    "num_iq_samples_dataset": NUM_IQ_SAMPLES,
    "num_signals_min": 1,
    "num_signals_max": 1,
    "fft_size": FFT_SIZE,
    "fft_stride": FFT_SIZE,
    "sample_rate": SAMPLE_RATE,
    "noise_power_db": 0.0,
    "snr_db_min": SNR_DB,
    "snr_db_max": SNR_DB,
    "cochannel_overlap_probability": 0.0,
    "signal_duration_in_samples_min": SIGNAL_DURATION_MIN,
    "signal_duration_in_samples_max": SIGNAL_DURATION_MAX,
    "bandwidth_min": SIGNAL_BANDWIDTH_MIN,
    "bandwidth_max": SIGNAL_BANDWIDTH_MAX,
    "signal_center_freq_min": -2500000,
    "signal_center_freq_max": 2499999,
    "frequency_min": -2500000,
    "frequency_max": 2499999,
}

### Support Functions

In [ ]:
def to_boxes(label):
    """Normalize a yolo_label target into an (N, 5) array of (cid, cx, cy, w, h)."""
    if label is None:
        return np.empty((0, 5))
    arr = np.asarray(label, dtype=float)
    if arr.ndim == 1:
        # single box (5,) -> (1,5); empty/degenerate -> (0,5)
        arr = arr.reshape(1, 5) if arr.size == 5 else arr.reshape(0, 5)
    return arr


def yolo_to_pixel_box(cx, cy, w, h, W, H):
    """Convert a normalized YOLO box to matplotlib imshow data coordinates.

    Time:  frames tile [0, num_iq) with no overlap, so frame j covers samples
           [j*N, (j+1)*N) and is centered at normalized time (j+0.5)/W.
           Inverting, normalized t -> column j = t*W - 0.5.

    Freq:  after fftshift + [::-1], row i holds frequency (H/2 - 1 - i)*fs/H,
           so row i = H/2 - 1 - f*H/fs. YOLOLabel emits y = 0.5 - f/fs,
           which assumes a symmetric [-fs/2, +fs/2] axis. Substituting gives
           row = y*H - 1.0.

    Returns ((x_lower_left, y_upper_left), width_px, height_px) for Rectangle.
    """
    x_center = cx * W - 0.5
    y_center = cy * H - 1.0
    box_w = w * W
    box_h = h * H
    return (x_center - box_w / 2.0, y_center - box_h / 2.0), box_w, box_h

### Dataset Generation

In [ ]:
ds = TorchSigIterableDataset(
    metadata=METADATA,
    component_transforms=[],
    transforms=[],
    target_labels=[],
    signal_generators=['zigbee'],
    validate_init=True
)
ds.seed(SEED)

# iq_samples signal
iq_samples = next(ds)   
print(iq_samples.shape)       

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.plot(iq_samples)

# im = ax.imshow(iq_samples, aspect="auto", origin='upper', 
#     cmap="viridis", 
#     vmin=ds.noise_power_db,
#     vmax=ds.noise_power_db + SNR_DB
# )
ax.set_title('zigbee')
ax.set_ylabel('Amplitude')
ax.set_xlabel('Time')
plt.show()


In [ ]:
# Generate a dataset with only Zigbee signals

ds = TorchSigIterableDataset(
    metadata=METADATA,
    component_transforms=[],
    transforms=[Spectrogram(fft_size=FFT_SIZE), YOLOLabel()],
    target_labels=["yolo_label"],   # yolo label format
    signal_generators=['zigbee'],
    validate_init=True
)
ds.seed(SEED)

# spec_db: wideband spectrogram in power dB
# label: single element list of (cid,cx,cy,w,h) 
spec_db, label = next(ds)          
H, W = spec_db.shape

In [ ]:
# generate figure
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(spec_db, aspect="auto", origin='upper', 
    cmap="viridis", 
    vmin=ds.noise_power_db,
    vmax=ds.noise_power_db + SNR_DB
)
ax.set_title('zigbee')
ax.set_ylabel('Frequency')
ax.set_xlabel('Time')
cbar = fig.colorbar(im, ax=ax, pad=0.02, extend="both")
cbar.set_label("Power (dB)")

# add bounding boxes
for cid, cx, cy, w, h in to_boxes(label):
    (x0, y0), box_w, box_h = yolo_to_pixel_box(cx, cy, w, h, W, H)
    ax.add_patch(Rectangle((x0, y0), box_w, box_h,
                    fill=False, edgecolor="red", lw=1.0, clip_on=True))

plt.show()